# 05 — LDA classification and evaluation

**MATLAB equivalent:** `LDAclassifier.m`

This notebook:
1. Normalises features using training-set mean and sample standard deviation.
2. Trains Linear Discriminant Analysis with uniform class priors.
3. Predicts the held-out repetitions.
4. Reports accuracy, confusion matrix, precision, recall, F1, balanced accuracy and macro F1.
5. Saves the model and normalisation parameters.

The standard deviation uses `ddof=1` to match MATLAB `std(X_train, 0, 1)`.

In [ ]:
from pathlib import Path
import numpy as np
import joblib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "NinaPro_DB5"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SUBJECTS = 10
N_CHANNELS = 16
FS = 200
WINDOW_SAMPLES = 40
STEP_SAMPLES = 10
CLASS_ORDER = np.arange(13)  # 0 = rest, 1–12 = movements

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

feature_data = joblib.load(OUTPUT_DIR / "feature_data.joblib")

X_train = feature_data["X_train"]
Y_train = feature_data["Y_train"]
X_test = feature_data["X_test"]
Y_test = feature_data["Y_test"]

## 1. Feature normalisation

Training statistics are applied unchanged to the test set. Features with zero training standard deviation are assigned a scale of 1, matching the MATLAB implementation.

In [ ]:
mu = np.mean(X_train, axis=0)
sigma = np.std(X_train, axis=0, ddof=1)

sigma[sigma == 0] = 1.0

X_train_norm = (X_train - mu) / sigma
X_test_norm = (X_test - mu) / sigma

assert np.all(np.isfinite(X_train_norm))
assert np.all(np.isfinite(X_test_norm))

print("Normalisation complete.")

## 2. Train LDA

Uniform priors are assigned to the classes present in the training set. The expected baseline contains all labels 0–12; the assertion makes missing training classes visible rather than silently changing the experiment.

In [ ]:
training_classes = np.unique(Y_train)
if not np.array_equal(training_classes, CLASS_ORDER):
    raise ValueError(
        "Expected all classes 0–12 in training. "
        f"Found: {training_classes.tolist()}"
    )

uniform_priors = np.full(len(training_classes), 1 / len(training_classes))

LDA_Model = LinearDiscriminantAnalysis(
    solver="lsqr",
    shrinkage=None,
    priors=uniform_priors,
)
LDA_Model.fit(X_train_norm, Y_train)

print("LDA training completed!")

## 3. Predict test dataset

In [ ]:
Y_pred = LDA_Model.predict(X_test_norm)

## 4. Overall accuracy

In [ ]:
accuracy = np.mean(Y_pred == Y_test) * 100
print(f"\\nOverall Accuracy: {accuracy:.2f} %")

## 5. Confusion matrix

In [ ]:
C = confusion_matrix(Y_test, Y_pred, labels=CLASS_ORDER)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=C, display_labels=CLASS_ORDER)
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
ax.set_title("NinaPro DB5 - LDA Classification")
plt.tight_layout()
plt.show()

## 6. Performance metrics

For a class present in the test set but never predicted, precision and F1 are set to zero. Such a class is retained in macro F1 rather than being omitted. Classes with no actual test samples are excluded from the macro averages.

In [ ]:
TP = np.diag(C).astype(float)
actual_count = C.sum(axis=1).astype(float)
predicted_count = C.sum(axis=0).astype(float)

precision = np.divide(
    TP, predicted_count,
    out=np.zeros_like(TP),
    where=predicted_count != 0
)
recall = np.divide(
    TP, actual_count,
    out=np.zeros_like(TP),
    where=actual_count != 0
)
F1 = np.divide(
    2 * precision * recall,
    precision + recall,
    out=np.zeros_like(TP),
    where=(precision + recall) != 0
)

present_in_test = actual_count > 0
balanced_accuracy = np.mean(recall[present_in_test])
macro_f1 = np.mean(F1[present_in_test])

print(f"Balanced Accuracy: {balanced_accuracy * 100:.2f} %")
print(f"Macro F1: {macro_f1 * 100:.2f} %")

## 7. Display results

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Movement": CLASS_ORDER,
    "TestWindows": actual_count.astype(int),
    "Precision": precision,
    "Recall": recall,
    "F1": F1,
})
display(results)

## 8. Save trained model and normalisation parameters

In [ ]:
model_artifact = {
    "model": LDA_Model,
    "mu": mu,
    "sigma": sigma,
    "class_order": CLASS_ORDER,
    "feature_order": feature_data["feature_order"],
    "sampling_frequency_hz": FS,
    "window_samples": WINDOW_SAMPLES,
    "step_samples": STEP_SAMPLES,
    "feature_names": (
        [f"MAV_ch{i}" for i in range(1, 17)]
        + [f"RMS_ch{i}" for i in range(1, 17)]
        + [f"WL_ch{i}" for i in range(1, 17)]
    ),
}

joblib.dump(model_artifact, OUTPUT_DIR / "NinaPro_DB5_LDA.joblib")
print("Model saved:", OUTPUT_DIR / "NinaPro_DB5_LDA.joblib")

## Reusing the model for a new window

The incoming window must have the same 40 × 16 shape and channel order as the training data.

In [ ]:
def predict_emg_window(window, artifact):
    features = extract_emg_features(window).reshape(1, -1)
    features_norm = (features - artifact["mu"]) / artifact["sigma"]
    prediction = artifact["model"].predict(features_norm)
    return int(prediction[0])

# Example:
# predicted_movement = predict_emg_window(new_window, model_artifact)